# 3. YOLO classification
Fine-tunes an Ultralytics YOLO classification model on the same split as notebook 02. The split is written out as train/val/test folders of cropped characters (the same crops the API produces). Flips are off because a mirrored MODI character is a different (or invalid) character.

`DATA`, `PER_CLASS` and `SEED` must match notebook 02.

In [1]:
pip install ultralytics

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, sys
from pathlib import Path

# run everything from the project root, whether Jupyter was started there or in notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))  # so `backend.ml.preprocessing` can be imported
print("project root:", ROOT)

project root: D:\Portfolio Projects\MODI Script Identification


In [3]:
# ---- settings ----
DATA = "data/processed/offline_subset"
PER_CLASS = 300
SEED = 42
WEIGHTS = "yolov8n-cls.pt"   # or yolo11n-cls.pt
IMGSZ = 64
EPOCHS = 15
BATCH = 64
WORKERS = 4
DEVICE = None                # e.g. 0 for GPU, "cpu"
WORK_DIR = "data/processed/yolo_cls"
OUT = "models/yolo_cls.pt"
REPORT_DIR = "reports/yolo_cls"

## Shared split
Same function in notebooks 02 to 05, so every method is trained and tested on the same source images when `DATA`, `PER_CLASS` and `SEED` match.

In [4]:
import cv2
import numpy as np
from sklearn.model_selection import train_test_split

# The crop function lives in the backend because the API must preprocess exactly like training.
from backend.ml.preprocessing import crop_square

EXTS = {".png", ".jpg", ".jpeg", ".bmp"}


def select_and_split(root, per_class=300, seed=42):
    """Returns (classes, paths, labels, {"train": idx, "val": idx, "test": idx}); 70/15/15, stratified.
    Deterministic: the same root, per_class and seed always give the same split."""
    root = Path(root)
    classes = sorted(d.name for d in root.iterdir() if d.is_dir())
    assert classes, f"no class folders found in {root}"
    rng = np.random.RandomState(seed)
    paths, labels = [], []
    for ci, c in enumerate(classes):
        files = sorted(f for f in (root / c).iterdir() if f.suffix.lower() in EXTS)
        if per_class and len(files) > per_class:
            keep = np.sort(rng.choice(len(files), per_class, replace=False))
            files = [files[i] for i in keep]
        paths.extend(files)
        labels.extend([ci] * len(files))
    labels = np.array(labels)
    idx = np.arange(len(paths))
    tr, tmp = train_test_split(idx, test_size=0.3, stratify=labels, random_state=seed)
    va, te = train_test_split(tmp, test_size=0.5, stratify=labels[tmp], random_state=seed)
    return classes, paths, labels, {"train": tr, "val": va, "test": te}


def load_crops(paths, size):
    """(N, size, size) uint8: tight-cropped, padded square, dark ink on white (same as the API)."""
    out = np.full((len(paths), size, size), 255, np.uint8)
    bad = 0
    for i, p in enumerate(paths):
        g = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if g is None:
            bad += 1
            continue
        out[i] = crop_square(g, size)
    if bad:
        print(f"warning: {bad} unreadable images were left blank")
    return out

## Write the shared split as image folders

In [5]:
import json, shutil, time

import cv2

classes, paths, labels, split = select_and_split(DATA, PER_CLASS, SEED)
crops = load_crops(paths, IMGSZ)
root = Path(WORK_DIR)
if root.exists():
    shutil.rmtree(root)
for name, ids in split.items():
    for n, i in enumerate(ids):
        d = root / name / classes[labels[i]]
        d.mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(d / f"{n:05d}.png"), crops[i])
sizes = {k: len(v) for k, v in split.items()}
print(f"classes={len(classes)} split sizes={sizes} -> {root}")

classes=57 split sizes={'train': 11970, 'val': 2565, 'test': 2565} -> data\processed\yolo_cls


## Train

In [6]:
from ultralytics import YOLO

model = YOLO(WEIGHTS)
model.train(data=str(root.resolve()), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
            workers=WORKERS, device=DEVICE, project="runs/yolo_cls", name="modi",
            exist_ok=True, patience=10, seed=SEED, plots=False,
            fliplr=0.0, flipud=0.0, hsv_h=0.0, hsv_s=0.0)

best = Path(model.trainer.save_dir) / "weights" / "best.pt"
Path(OUT).parent.mkdir(parents=True, exist_ok=True)
shutil.copy(best, OUT)

Ultralytics 8.4.191  Python-3.14.3 torch-2.14.0+cpu CPU (Intel Core 5 120U)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\Portfolio Projects\MODI Script Identification\data\processed\yolo_cls, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.4, imgsz=64, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=modi, nbs=64, nms=None, o

'models/yolo_cls.pt'

## Evaluate on the test split

In [7]:
final = YOLO(OUT)
m = final.val(data=str(root.resolve()), split="test", imgsz=IMGSZ, device=DEVICE, plots=False)

sample = cv2.imread(str(next((root / "test").rglob("*.png"))))
for _ in range(5):
    final.predict(sample, imgsz=IMGSZ, verbose=False)
t = time.perf_counter()
for _ in range(50):
    final.predict(sample, imgsz=IMGSZ, verbose=False)
ms = (time.perf_counter() - t) / 50 * 1000

metrics = {"top1_accuracy": float(m.top1), "top5_accuracy": float(m.top5),
           "inference_ms_per_image": ms, "num_classes": len(classes), "split_sizes": sizes,
           "weights": WEIGHTS, "imgsz": IMGSZ}
Path(REPORT_DIR).mkdir(parents=True, exist_ok=True)
(Path(REPORT_DIR) / "metrics.json").write_text(json.dumps(metrics, indent=2))
print("TEST metrics:", metrics)
print(f"saved {OUT}")

Ultralytics 8.4.191  Python-3.14.3 torch-2.14.0+cpu CPU (Intel Core 5 120U)
YOLOv8n-cls summary (fused): 30 layers, 1,507,897 parameters, 0 gradients, 3.3 GFLOPs
train: D:\Portfolio Projects\MODI Script Identification\data\processed\yolo_cls\train... found 11970 images in 57 classes  
val: D:\Portfolio Projects\MODI Script Identification\data\processed\yolo_cls\val... found 2565 images in 57 classes  
test: D:\Portfolio Projects\MODI Script Identification\data\processed\yolo_cls\test... found 2565 images in 57 classes  
test: Fast image access  (ping: 0.3±0.3 ms, read: 1.8±0.7 MB/s, size: 1.7 KB)
test: Scanning D:\Portfolio Projects\MODI Script Identification\data\processed\yolo_cls\test... 2565 images, 0 corrupt: 100% ━━━━━━━━━━━━ 2565/2565 1.4Kit/s 1.9s0.0s
test: New cache created: D:\Portfolio Projects\MODI Script Identification\data\processed\yolo_cls\test.cache
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 161/161 8.6it/s 18.8s0.1s
                   all      0.8